# CudaSIFT + RootSIFT (GPU) ORB-SLAM3 fork -- KITTI whole-set sweep (Kaggle GPU)

Runs `orbslam3_sift_kitti_ate` (the SIFT-feature fork of ORB-SLAM3, GPU-accelerated at
the extractor via [Celebrandil/CudaSift](https://github.com/Celebrandil/CudaSift) +
RootSIFT) against KITTI odometry sequences 00-05, sweeping runtime toggles for the
tracker (KLT vs SQPnP) and the epipolar-bridge recovery mechanism (which keeps a map
alive across a tracking dropout instead of resetting -- see
`ORBSLAM3_SIFT_EPIPOLAR_BRIDGE_PLAN.md` in the repo for the full writeup).

**Before running:**
1. Notebook settings -> **Accelerator: GPU** (T4 x2 or P100). **Internet: On**.
2. **Add Data** -> attach a KITTI odometry (grayscale) Kaggle Dataset, e.g.
   `xuehu12/kitti-odometry-gray` (images only -- ground-truth poses come from this
   repo's own `kitti_poses/`, not the dataset; `kaggle/setup_and_run.sh` handles that
   fallback automatically).
3. `REPO_URL` below must point at your pushed repo (public, or bake a token into the
   URL) -- the commits this sweep depends on (epipolar bridge, SQPnP tracker,
   mono-init stale-reference fix, the `TRACKER`/`BRIDGE`/`INITFIX` runtime toggles,
   and the dataset auto-detect fix) must already be pushed to `main`.

**What the toggles are** (env vars, read once at startup, no rebuild needed to change
them -- see `Tracking::trackerUsesSqpnp()`/`bridgeMode()`/`initFixEnabled()`):
- `TRACKER=klt|sqpnp` -- per-frame tracker (default sqpnp).
- `BRIDGE=off|rl|lost|both` -- epipolar-bridge recovery: `rl` = only in the
  RECENTLY_LOST branch (established maps losing track), `lost` = only in the LOST
  branch (young maps, <10 KFs, that would otherwise reset immediately -- this is
  where most of Kaggle's first-run resets came from), `both` = try it in either
  (default).
- `INITFIX=on|off` -- advance the mono-init reference once matched disparity exceeds
  150px, instead of retrying forever against a frozen stale reference (fixes seq01's
  highway degenerate-init problem; default on).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader


In [ ]:
import os

REPO_URL = "https://github.com/nguyenhuunam852/ORB-SLAM-SIFT.git"  # set to your pushed repo
REPO_DIR = "/kaggle/working/ORB-SLAM-SIFT"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already present, skipping clone (rm -rf it to force a fresh clone)")


## Build + smoke test

One-time: apt deps, g2o build, CudaSift clone, ONNX Runtime GPU download, cmake
configure + build (`USE_CUDASIFT=1`). Safe to rerun (skips steps whose output already
exists). Ends with a default 1000-frame run on whichever sequence it auto-detects
under `/kaggle/input` -- if this cell errors with *"Could not find a KITTI sequence
directory"*, the dataset isn't attached yet (see step 2 above), not a code problem.
Read the printed `image_0 candidates` list if it picks the wrong sequence.

In [ ]:
!cd {REPO_DIR} && bash kaggle/setup_and_run.sh


## Main sweep: 6 configs x sequences 00-05

`SKIP_BUILD=1` reuses the build above -- no rebuild between configs, since
`TRACKER`/`BRIDGE`/`INITFIX` are pure runtime env-var toggles. Edit `SEQS` (e.g.
`["00"]`) or `CONFIGS` to narrow the sweep; each (config, seq) pair runs the full
sequence (`MAX_FRAMES=0`) and its stdout is captured to
`/kaggle/working/<config>_seq<NN>.log` for later inspection.

**Configs swept:**
- `baseline_klt_nobridge` -- pre-recipe behavior (KLT, no bridge, no init-fix): the
  number every other row needs to beat.
- `klt_bridge_both` -- isolates the bridge's effect on top of the original KLT
  tracker.
- `sqpnp_nobridge` -- isolates SQPnP's own effect on seq00 (an open question from the
  first Kaggle run: is high reset count from SQPnP itself, or from the bridge being
  bypassed by young-map deaths?).
- `sqpnp_bridge_rl` / `sqpnp_bridge_lost` / `sqpnp_bridge_both` -- isolates which
  bridge branch (established-map loss vs young-map rescue) actually helps.

In [ ]:
import os, re, glob, subprocess

SEQS = [f"{i:02d}" for i in range(0, 6)]   # 00..05 -- edit to e.g. ["00"] for a single seq

# name -> (TRACKER, BRIDGE, INITFIX)
CONFIGS = {
    "baseline_klt_nobridge":  ("klt",   "off",  "off"),
    "klt_bridge_both":        ("klt",   "both", "on"),
    "sqpnp_nobridge":         ("sqpnp", "off",  "on"),
    "sqpnp_bridge_rl":        ("sqpnp", "rl",   "on"),
    "sqpnp_bridge_lost":      ("sqpnp", "lost", "on"),
    "sqpnp_bridge_both":      ("sqpnp", "both", "on"),
}

cands = sorted(glob.glob("/kaggle/input/**/sequences", recursive=True))
SEQ_BASE = cands[0] if cands else "/kaggle/input/kitti-odometry-gray/dataset/sequences"
print("SEQ_BASE =", SEQ_BASE, "  (override manually if wrong)")

rows = []
for cfg_name, (tracker, bridge, initfix) in CONFIGS.items():
    for seq in SEQS:
        seq_dir = f"{SEQ_BASE}/{seq}"
        poses   = f"{REPO_DIR}/kitti_poses/{seq}.txt"
        if not os.path.isdir(f"{seq_dir}/image_0") or not os.path.isfile(poses):
            print(f"[skip {cfg_name}/{seq}] missing images or poses"); continue
        logf = f"/kaggle/working/{cfg_name}_seq{seq}.log"
        env = os.environ.copy()
        env.update(SKIP_BUILD="1", USE_CUDASIFT="1", START_FRAME="0", MAX_FRAMES="0",
                   KITTI_SEQ_DIR=seq_dir, KITTI_POSES=poses,
                   OUT_PREFIX=f"/kaggle/working/{cfg_name}_seq{seq}",
                   TRACKER=tracker, BRIDGE=bridge, INITFIX=initfix)
        print(f"=== {cfg_name} / seq{seq} (TRACKER={tracker} BRIDGE={bridge} INITFIX={initfix}) ===", flush=True)
        with open(logf, "w") as f:
            subprocess.run(["bash", "kaggle/setup_and_run.sh"], cwd=REPO_DIR, env=env,
                           stdout=f, stderr=subprocess.STDOUT)
        txt = open(logf, errors="ignore").read()
        frags = re.findall(r"pathLen=([\d.]+)m ATE\(rmse[^=]*=([\d.]+)/", txt)
        cov_m = sum(float(p) for p, _ in frags)
        m = re.search(r"GT path length:\s*([\d.]+)", txt)
        gt_m = float(m.group(1)) if m else 0.0
        worst = max((float(a) for _, a in frags), default=0.0)
        resets = len(re.findall(r"A new map is started|Reseting current map", txt))
        bridge_ok = len(re.findall(r"\[bridge\].*SUCCESS", txt))
        ninit  = len(re.findall(r"mono-init.*SUCCESS", txt))
        nfail  = len(re.findall(r"reconstruct FAILED", txt))
        initp  = 100.0 * ninit / max(1, ninit + nfail)
        covp   = 100.0 * cov_m / gt_m if gt_m else 0.0
        rows.append((cfg_name, seq, cov_m, covp, worst, resets, bridge_ok, initp, len(frags)))
        print(f"    cov={cov_m:.1f}m/{covp:.1f}%  worstATE={worst:.2f}m  resets={resets}  "
              f"bridgeOK={bridge_ok}  init%={initp:.1f}  frags={len(frags)}", flush=True)

print("\n\n=== RESULTS TABLE (toggle sweep, seq 00-05) ===\n")
print("| Config | Seq | Cov(m) | Cov% | worstATE | Resets | BridgeOK | Init% | Frags |")
print("|--------|-----|--------|------|----------|--------|----------|-------|-------|")
for cfg, seq, cm, cp, wa, rs, br, ip, nf in rows:
    print(f"| {cfg} | {seq} | {cm:.1f} | {cp:.1f}% | {wa:.2f}m | {rs} | {br} | {ip:.1f}% | {nf} |")


## Reading the table

- **`baseline_klt_nobridge`** is the number every other row must beat -- coverage and
  reset count with none of this session's changes active.
- Compare **`sqpnp_nobridge`** against baseline to isolate whether SQPnP itself helps
  or hurts seq00 independent of the bridge.
- Compare **`sqpnp_bridge_rl`** vs **`sqpnp_bridge_lost`** vs **`sqpnp_bridge_both`** to
  see which bridge branch (established-map loss vs young-map rescue) is doing the
  work -- the first Kaggle run (no LOST-path bridge yet) fired only 45x against 317
  resets, because most maps died young (<10 KFs) and bypassed the RECENTLY_LOST branch
  entirely.
- Per-sequence logs are at `/kaggle/working/<config>_seq<NN>.log` if a row needs
  closer inspection (grep `[bridge]`, `[mono-init]`, `Fail to track local map`).